In [0]:
df_customer=spark.read.format("delta")\
    .load("abfss://data@storagechinmayi.dfs.core.windows.net/silver/customers")

In [0]:
df_customer.display()

customer_id,name,email,city,state,join_date,loyalty_flag
C0001,Robin Smith,maywilliam@hendrix.com,Davisville,Texas,2023-09-21,false
C0002,Troy Wilson,yallen@hotmail.com,East Hannah,Alabama,2024-11-06,false
C0003,Brandon West,adamslisa@clay.com,Lake Shannon,Georgia,2024-10-20,false
C0004,William Rosario,vanessaosborne@morris.biz,Armstrongfurt,Georgia,2022-11-24,false
C0005,William Arnold,dhuffman@mooney.com,South Dawnside,Texas,2024-03-21,false
C0006,Jennifer Jefferson,randylutz@hotmail.com,Port Isabel,Florida,2023-09-10,false
C0007,Darius Donaldson,danielle74@stafford.com,East Deborah,Mississippi,2025-03-04,true
C0008,Jennifer Webb,berryjohn@gmail.com,Edwardsside,Idaho,2023-01-21,false
C0009,Jennifer Campos,mortonjonathan@hotmail.com,South Sean,Florida,2024-11-27,false
C0010,Lucas Cruz,morgan78@hotmail.com,New Jasminemouth,California,2023-01-25,true


In [0]:
df_order=spark.read.format("delta")\
    .load("abfss://data@storagechinmayi.dfs.core.windows.net/silver/orders")

In [0]:
df_order.display()

customer_id,product_id,order_id,quantity,order_date,price,total_value,loyalty_flag,discount
C0094,P0211,O00001,5,2025-06-05,59.35,296.75,false,296.75
C0009,P0626,O00002,3,2024-07-03,234.3,702.9000000000001,false,702.9
C0039,P0217,O00003,1,2024-06-29,461.23,461.23,false,461.23
C0608,P0299,O00004,3,2025-03-13,314.74,944.22,false,944.22
C0544,P0055,O00005,1,2024-12-02,282.22,282.22,true,253.998
C0634,P0190,O00006,4,2024-07-08,109.04,436.16,false,436.16
C0610,P0613,O00007,5,2025-04-22,431.26,2156.3,true,1940.67
C0214,P0476,O00008,2,2025-05-04,477.52,955.04,true,859.536
C0757,P0695,O00009,2,2023-11-13,418.53,837.06,true,753.354
C0918,P0450,O00010,1,2024-05-24,293.52,293.52,true,264.168


In [0]:
df_product=spark.read.format("delta")\
    .load("abfss://data@storagechinmayi.dfs.core.windows.net/silver/products")

In [0]:
df_flat_survey=spark.read.format("delta")\
    .load("abfss://data@storagechinmayi.dfs.core.windows.net/silver/survey")

In [0]:
DimCustomer=df_customer.select("customer_id", "name", "city", "loyalty_flag")

In [0]:
DimProduct=df_product.select("product_id", "product_name", "category", "price")
DimProduct=DimProduct.withColumnRenamed("category", "product_category")

In [0]:
DimProduct.display()

product_id,product_name,product_category,price
P0001,But,Books,326.01
P0002,Step,Electronics,17.79
P0003,Line,Toys,72.14
P0004,Management,Toys,217.22
P0005,Again,Electronics,352.71
P0006,Drive,Books,146.94
P0007,Production,Toys,354.93
P0008,Leg,Clothing,177.48
P0009,While,Electronics,162.94
P0010,Cultural,Electronics,312.19


In [0]:
df_order=df_order.join(df_flat_survey.select("is_recommended","order_id"),"order_id", "left")

In [0]:
Fact_Sales=df_order.select("order_id", "customer_id", "product_id", "price", "total_value", "loyalty_flag", "discount")


In [0]:
DimCustomer.write.mode("parquet")\
    .mode("overwrite")\
    .option("path", "abfss://data@storagechinmayi.dfs.core.windows.net/gold/DimCustomer")\
    .saveAsTable("DimCustomer")

In [0]:
DimProduct.write.mode("parquet")\
    .mode("overwrite")\
    .option("path", "abfss://data@storagechinmayi.dfs.core.windows.net/gold/DimProduct")\
    .saveAsTable("DimProduct")

In [0]:
Fact_Sales.write.mode("parquet")\
    .mode("overwrite")\
    .option("path", "abfss://data@storagechinmayi.dfs.core.windows.net/gold/Fact_Sales")\
    .saveAsTable("Fact_Sales")